In [1]:
# pip install numpy
# pip install nltk
# pip install pythainlp
# pip install attacut

In [1]:
import numpy as np
import pandas as pd 
from nltk.util import ngrams
from collections import Counter
from pythainlp.tokenize import word_tokenize

In [2]:
class Trigram : 
    
    def __init__(self, document, tokenize, n_gram=3):
        self.document = document
        self.tokenize = tokenize
        self.n_gram = n_gram
        self.mdm_terms_ngrams = [self.generate_ngrams(text=term, n=self.n_gram) for term in self.document]
        self.mdm_terms_vectors = [Counter(term_ngrams) for term_ngrams in self.mdm_terms_ngrams]
        self.mdm_terms_ngrams = None
        
    def generate_ngrams(self, text, n=3):
        return list(ngrams(text, n))

    def cosine_similarity_vectors(self, vec1, vec2):
        dot_product = sum(vec1[term] * vec2.get(term, 0) for term in vec1)
        norm1 = np.sqrt(sum(v**2 for v in vec1.values()))
        norm2 = np.sqrt(sum(v**2 for v in vec2.values()))
        return dot_product / (norm1 * norm2) if norm1 and norm2 else 0.0

    def search(self, query):
        query_result = []
        similarity_scores = []
        query_ngrams = self.generate_ngrams(query, self.n_gram)
        query_vector = Counter(query_ngrams)
        for term_vector in self.mdm_terms_vectors:
            score = self.cosine_similarity_vectors(query_vector, term_vector)
            similarity_scores.append(score)
        ranked_terms = sorted(
            enumerate(
                similarity_scores
            ), 
            key=lambda x: x[1], 
            reverse=True
        )
        for index, score in ranked_terms : 
            query_result.append(
                {
                    'idx': index, 
                    'query': self.document[index], 
                    'score': score
                }
            )
        return query_result

def attacut(txt):
    return word_tokenize(txt, engine='attacut', keep_whitespace=False)

In [3]:
documents = [
    "The cat sat on the mat.",
    "The dog barked loudly.",
    "The quick brown fox jumps over the lazy dog.",
    "A fox is a quick, cunning animal.",
    "Dogs are loyal and intelligent pets."
]

In [4]:
class_trigram = Trigram(document=documents, tokenize=attacut)

In [5]:
query = 'cat and mat'

In [6]:
query_result = class_trigram.search(query=query)
query_result

[{'idx': 0, 'query': 'The cat sat on the mat.', 'score': 0.3333333333333333},
 {'idx': 4,
  'query': 'Dogs are loyal and intelligent pets.',
  'score': 0.17149858514250882},
 {'idx': 3,
  'query': 'A fox is a quick, cunning animal.',
  'score': 0.05986843400892498},
 {'idx': 1, 'query': 'The dog barked loudly.', 'score': 0.0},
 {'idx': 2,
  'query': 'The quick brown fox jumps over the lazy dog.',
  'score': 0.0}]

In [7]:
pd.DataFrame(query_result)

,idx,query,score
0,0,The cat sat on the mat.,0.333333
1,4,Dogs are loyal and intelligent pets.,0.171499
2,3,"A fox is a quick, cunning animal.",0.059868
3,1,The dog barked loudly.,0.000000
4,2,The quick brown fox jumps over the lazy dog.,0.000000


# Not Class example

In [8]:
def generate_ngrams(text, n=3):
    return list(ngrams(text, n))

def cosine_similarity_vectors(vec1, vec2):
    dot_product = sum(vec1[term] * vec2.get(term, 0) for term in vec1)
    norm1 = np.sqrt(sum(v**2 for v in vec1.values()))
    norm2 = np.sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2) if norm1 and norm2 else 0.0

In [9]:
documents = [
    "The cat sat on the mat.",
    "The dog barked loudly.",
    "The quick brown fox jumps over the lazy dog.",
    "A fox is a quick, cunning animal.",
    "Dogs are loyal and intelligent pets."
]

In [10]:
similarity_scores = []
mdm_terms_ngrams = [generate_ngrams(text=term) for term in documents]
mdm_terms_vectors = [Counter(term_ngrams) for term_ngrams in mdm_terms_ngrams]

query = 'fox so quick'
query_ngrams = generate_ngrams(query, 3)
query_vector = Counter(query_ngrams)
for term_vector in mdm_terms_vectors:
    score = cosine_similarity_vectors(query_vector, term_vector)
    similarity_scores.append(score)

ranked_terms = sorted(
    enumerate(
        similarity_scores
    ), 
    key=lambda x: x[1], 
    reverse=True
)

query_res = []
for index, score in ranked_terms : 
    query_res.append(
        {
            'idx': index, 
            'query': documents[index], 
            'score': score
        }
    )

query_res

[{'idx': 3,
  'query': 'A fox is a quick, cunning animal.',
  'score': 0.3407771005482389},
 {'idx': 2,
  'query': 'The quick brown fox jumps over the lazy dog.',
  'score': 0.28603877677367767},
 {'idx': 0, 'query': 'The cat sat on the mat.', 'score': 0.0},
 {'idx': 1, 'query': 'The dog barked loudly.', 'score': 0.0},
 {'idx': 4, 'query': 'Dogs are loyal and intelligent pets.', 'score': 0.0}]